rain/validation/test split

In [1]:
import random 
import math 


def train_val_test_split(X,y,train_ratio=0.6,val_ratio=0.2,seed=42):
    random.seed(seed)
    n=len(X)
    indices=list(range(n))
    random.shuffle(indices)
    
    train_end=int(n*train_ratio)
    val_end=int(n*(train_ratio+val_ratio))
    train_idx = indices[:train_end]
    val_idx = indices[train_end:val_end]
    test_idx = indices[val_end:]

    X_train = [X[i] for i in train_idx]
    y_train = [y[i] for i in train_idx]
    X_val = [X[i] for i in val_idx]
    y_val = [y[i] for i in val_idx]
    X_test = [X[i] for i in test_idx]
    y_test = [y[i] for i in test_idx]

    return X_train, y_train, X_val, y_val, X_test, y_test

In [3]:
X = list(range(10))
y = [x * x for x in X]

X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X, y)
print(X,y)
print("Train X:", X_train)
print("Train y:", y_train)
print("Val   X:", X_val)
print("Val   y:", y_val)
print("Test  X:", X_test)
print("Test  y:", y_test)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9] [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
Train X: [7, 3, 2, 8, 5, 6]
Train y: [49, 9, 4, 64, 25, 36]
Val   X: [9, 4]
Val   y: [81, 16]
Test  X: [0, 1]
Test  y: [0, 1]


K-fold and stratified K-fold cross-validation

In [5]:
def kfold_split(n, k=5, seed=42):
    random.seed(seed)
    indices = list(range(n))
    random.shuffle(indices)

    fold_size = n // k
    folds = []

    for i in range(k):
        start = i * fold_size
        end = start + fold_size if i < k - 1 else n
        val_idx = indices[start:end]
        train_idx = indices[:start] + indices[end:]
        folds.append((train_idx, val_idx))

    return folds


def stratified_kfold_split(y, k=5, seed=42):
    random.seed(seed)

    class_indices = {}
    for i, label in enumerate(y):
        class_indices.setdefault(label, []).append(i)

    for label in class_indices:
        random.shuffle(class_indices[label])

    folds = [{"train": [], "val": []} for _ in range(k)]

    for label, indices in class_indices.items():
        fold_size = len(indices) // k
        for i in range(k):
            start = i * fold_size
            end = start + fold_size if i < k - 1 else len(indices)
            val_part = indices[start:end]
            train_part = indices[:start] + indices[end:]
            folds[i]["val"].extend(val_part)
            folds[i]["train"].extend(train_part)

    return [(f["train"], f["val"]) for f in folds]


def cross_validate(X, y, model_fn, k=5, metric_fn=None, stratified=False):
    n = len(X)

    if stratified:
        folds = stratified_kfold_split(y, k)
    else:
        folds = kfold_split(n, k)

    scores = []
    for train_idx, val_idx in folds:
        X_train = [X[i] for i in train_idx]
        y_train = [y[i] for i in train_idx]
        X_val = [X[i] for i in val_idx]
        y_val = [y[i] for i in val_idx]

        model = model_fn()
        model.fit(X_train, y_train)
        predictions = [model.predict(x) for x in X_val]

        if metric_fn:
            score = metric_fn(y_val, predictions)
        else:
            score = sum(1 for yt, yp in zip(y_val, predictions) if yt == yp) / len(y_val)
        scores.append(score)

    return scores

In [6]:
folds = kfold_split(n=10, k=5, seed=42)
assert len(folds) == 5
for train_idx, val_idx in folds:
    assert len(val_idx) == 2
    assert len(train_idx) == 8
    assert set(train_idx).isdisjoint(val_idx)
all_val = [i for _, val in folds for i in val]
assert sorted(all_val) == list(range(10))
assert kfold_split(10, 5, seed=42) == folds

In [7]:
y = [0] * 6 + [1] * 4
folds = stratified_kfold_split(y, k=2, seed=42)
assert len(folds) == 2
for train_idx, val_idx in folds:
    assert set(train_idx).isdisjoint(val_idx)
    labels = [y[i] for i in val_idx]
    assert 0 in labels and 1 in labels
all_val = [i for _, val in folds for i in val]
assert sorted(all_val) == list(range(len(y)))

In [8]:
X = [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9]]
y = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]

class ThresholdModel:
    def fit(self, X, y):
        self.threshold = sum(x[0] for x in X) / len(X)
    def predict(self, x):
        return 1 if x[0] > self.threshold else 0

def accuracy(a, b):
    return sum(x == y for x, y in zip(a, b)) / len(a)

scores = cross_validate(X, y, lambda: ThresholdModel(), k=5, metric_fn=accuracy)
assert len(scores) == 5
assert all(0.0 <= s <= 1.0 for s in scores)
assert scores == cross_validate(X, y, lambda: ThresholdModel(), k=5, metric_fn=accuracy)

Confusion matrix and classification metrics

In [9]:
def confusion_matrix(y_true, y_pred):
    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)
    return tp, tn, fp, fn


def accuracy(y_true, y_pred):
    tp, tn, fp, fn = confusion_matrix(y_true, y_pred)
    total = tp + tn + fp + fn
    return (tp + tn) / total if total > 0 else 0.0


def precision(y_true, y_pred):
    tp, tn, fp, fn = confusion_matrix(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def recall(y_true, y_pred):
    tp, tn, fp, fn = confusion_matrix(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def roc_curve(y_true, y_scores):
    thresholds = sorted(set(y_scores), reverse=True)
    tpr_list = []
    fpr_list = []

    total_positives = sum(y_true)
    total_negatives = len(y_true) - total_positives

    for threshold in thresholds:
        y_pred = [1 if s >= threshold else 0 for s in y_scores]
        tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
        fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)

        tpr = tp / total_positives if total_positives > 0 else 0.0
        fpr = fp / total_negatives if total_negatives > 0 else 0.0

        tpr_list.append(tpr)
        fpr_list.append(fpr)

    return fpr_list, tpr_list, thresholds


def auc_roc(y_true, y_scores):
    fpr_list, tpr_list, _ = roc_curve(y_true, y_scores)

    pairs = sorted(zip(fpr_list, tpr_list))
    fpr_sorted = [p[0] for p in pairs]
    tpr_sorted = [p[1] for p in pairs]

    area = 0.0
    for i in range(1, len(fpr_sorted)):
        width = fpr_sorted[i] - fpr_sorted[i - 1]
        height = (tpr_sorted[i] + tpr_sorted[i - 1]) / 2
        area += width * height

    return area

In [10]:
# ---- shared toy data ----
y_true = [1, 0, 1, 1, 0, 0, 1, 0]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0]
# tp=3, tn=3, fp=1, fn=1

# ---- confusion_matrix ----
tp, tn, fp, fn = confusion_matrix(y_true, y_pred)
print("confusion_matrix:", (tp, tn, fp, fn))
assert (tp, tn, fp, fn) == (3, 3, 1, 1)

# ---- accuracy ----
acc = accuracy(y_true, y_pred)
print("accuracy:", acc)
assert acc == 6 / 8

# ---- precision ----
prec = precision(y_true, y_pred)
print("precision:", prec)
assert prec == 3 / 4

# ---- recall ----
rec = recall(y_true, y_pred)
print("recall:", rec)
assert rec == 3 / 4

# ---- f1_score ----
f1 = f1_score(y_true, y_pred)
print("f1_score:", f1)
assert abs(f1 - 0.75) < 1e-9

# ---- edge case: no positives predicted / empty ----
print("precision (no positive preds):", precision([0, 0], [0, 0]))
print("recall (no positives true):", recall([0, 0], [0, 0]))
print("accuracy (empty):", accuracy([], []))

# ---- roc_curve ----
y_scores = [0.9, 0.1, 0.8, 0.4, 0.2, 0.6, 0.7, 0.3]
fpr, tpr, thr = roc_curve(y_true, y_scores)
print("roc_curve fpr:", fpr)
print("roc_curve tpr:", tpr)
print("roc_curve thresholds:", thr)
assert len(fpr) == len(tpr) == len(thr)

# ---- auc_roc ----
auc = auc_roc(y_true, y_scores)
print("auc_roc:", auc)
assert 0.0 <= auc <= 1.0

# perfect separator -> auc == 1.0
perfect_scores = [0.9, 0.1, 0.8, 0.7, 0.2, 0.3, 0.6, 0.4]
auc_perfect = auc_roc(y_true, perfect_scores)
print("auc_roc (perfect):", auc_perfect)
assert abs(auc_perfect - 1.0) < 1e-9

confusion_matrix: (3, 3, 1, 1)
accuracy: 0.75
precision: 0.75
recall: 0.75
f1_score: 0.75
precision (no positive preds): 0.0
recall (no positives true): 0.0
accuracy (empty): 0.0
roc_curve fpr: [0.0, 0.0, 0.0, 0.25, 0.25, 0.5, 0.75, 1.0]
roc_curve tpr: [0.25, 0.5, 0.75, 0.75, 1.0, 1.0, 1.0, 1.0]
roc_curve thresholds: [0.9, 0.8, 0.7, 0.6, 0.4, 0.3, 0.2, 0.1]
auc_roc: 0.9375
auc_roc (perfect): 1.0


Regression metrics

In [11]:
def mse(y_true, y_pred):
    n = len(y_true)
    return sum((yt - yp) ** 2 for yt, yp in zip(y_true, y_pred)) / n


def rmse(y_true, y_pred):
    return math.sqrt(mse(y_true, y_pred))


def mae(y_true, y_pred):
    n = len(y_true)
    return sum(abs(yt - yp) for yt, yp in zip(y_true, y_pred)) / n


def r_squared(y_true, y_pred):
    mean_y = sum(y_true) / len(y_true)
    ss_res = sum((yt - yp) ** 2 for yt, yp in zip(y_true, y_pred))
    ss_tot = sum((yt - mean_y) ** 2 for yt in y_true)
    if ss_tot == 0:
        return 0.0
    return 1.0 - ss_res / ss_tot